In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# gate frazionari

<details>
<summary><b>Versioni dei pacchetti</b></summary>

Il codice in questa pagina è stato sviluppato utilizzando i seguenti requisiti.
Si raccomanda di usare queste versioni o versioni più recenti.

```
qiskit[all]~=2.3.0
qiskit-ibm-runtime~=0.43.1
```
</details>

Questa pagina presenta due nuovi tipi di gate supportati nel parco QPU di IBM Quantum&reg;. Questi gate *frazionari* sono supportati sulle [QPU Heron](/guides/processor-types#heron) nelle seguenti forme:
- $R_{ZZ}(\theta)$ per $0 \lt \theta \leq \pi/2$
- $R_X(\theta)$ per qualsiasi $\theta$

Questa pagina tratta i casi d'uso in cui l'impiego dei gate frazionari può migliorare l'efficienza dei tuoi flussi di lavoro, oltre a spiegare come utilizzarli sulle QPU IBM Quantum.

## Come usare i gate frazionari
Internamente, questi gate frazionari funzionano eseguendo direttamente una rotazione $R_{ZZ}(\theta)$ e $R_X(\theta)$ per un angolo arbitrario. L'uso del gate $R_X(\theta)$ può ridurre la durata e l'errore per le rotazioni a qubit singolo di angolo arbitrario fino a un fattore due. L'esecuzione diretta della rotazione del gate $R_{ZZ}(\theta)$ evita la decomposizione in più oggetti `CZGate`, riducendo in modo analogo la durata e l'errore del circuito. Questo è particolarmente utile per i circuiti che contengono molte rotazioni a uno e due qubit, come quando si simulano le dinamiche di un sistema quantistico o quando si usa un ansatz variazionale con molti parametri.

Sebbene questi tipi di gate siano presenti nella [libreria di gate standard](./circuit-library) che un `QuantumCircuit` può possedere, possono essere usati solo su specifiche QPU IBM Quantum, che devono essere caricate con il flag `use_fractional_gates` impostato a `True` (come mostrato di seguito). Questo flag garantirà che i gate frazionari siano inclusi nel `Target` del backend per il transpiler.

In [1]:
from qiskit import QuantumCircuit
from qiskit.circuit import Parameter
from qiskit.transpiler import generate_preset_pass_manager
from qiskit.visualization.timeline import draw as draw_timeline, IQXSimple

from qiskit_ibm_runtime import QiskitRuntimeService


num_qubits = 5
num_time_steps = 3
rx_angle = 0.1
rzz_angle = 0.1

ising_circuit = QuantumCircuit(num_qubits)
for i in range(num_time_steps):
    # rx layer
    for q in range(num_qubits):
        ising_circuit.rx(rx_angle, q)
    for q in range(1, num_qubits - 1, 2):
        ising_circuit.rzz(rzz_angle, q, q + 1)
    # 2nd rzz layer
    for q in range(0, num_qubits - 1, 2):
        ising_circuit.rzz(rzz_angle, q, q + 1)
    ising_circuit.barrier()
ising_circuit.draw("mpl")

<Image src="../docs/images/guides/fractional-gates/extracted-outputs/9cb7f696-dec0-4393-9320-fe945326c893-0.svg" alt="Output of the previous code cell" />

Questo esempio di codice mostra come usare i gate frazionari nel contesto di un flusso di lavoro che simula le dinamiche di una catena di Ising usando i gate frazionari. La durata del circuito viene poi confrontata con un backend che non usa gate frazionari.

> **Note:** Il valore di errore riportato nel `Target` di un backend con gate frazionari abilitati è semplicemente una copia dell'analogo gate non frazionario (che potrebbe non essere lo stesso). Questo perché la comunicazione dei tassi di errore sui gate frazionari non è ancora supportata.

  Tuttavia, poiché il tempo di gate dei gate frazionari rispetto a quelli non frazionari è lo stesso, è ragionevole assumere che i loro tassi di errore siano comparabili — soprattutto quando la sorgente di errore dominante in un circuito è dovuta al rilassamento.

In [2]:
service = QiskitRuntimeService()
backend_fractional = service.backend("ibm_torino", use_fractional_gates=True)
backend_conventional = service.backend(
    "ibm_torino", use_fractional_gates=False
)

pm_fractional = generate_preset_pass_manager(
    optimization_level=3, backend=backend_fractional, scheduling_method="alap"
)
pm_conventional = generate_preset_pass_manager(
    optimization_level=3,
    backend=backend_conventional,
    scheduling_method="alap",
)

ising_circuit_fractional = pm_fractional.run(ising_circuit)
ising_circuit_conventional = pm_conventional.run(ising_circuit)

![Output della cella di codice precedente](../docs/images/guides/fractional-gates/extracted-outputs/9cb7f696-dec0-4393-9320-fe945326c893-0.svg)

Specifica due oggetti backend: uno con i gate frazionari abilitati e l'altro con essi disabilitati, poi esegui il transpiling di entrambi.

In [3]:
# Draw timeline of circuit with conventional gates
draw_timeline(
    ising_circuit_conventional,
    idle_wires=False,
    target=backend_conventional.target,
    time_range=(0, 500),
    style=IQXSimple(),
)

<Image src="../docs/images/guides/fractional-gates/extracted-outputs/0013f5fa-4072-4aa4-94fe-7e195435f828-0.svg" alt="Output of the previous code cell" />

In [4]:
# Draw timeline of circuit with fractional gates
draw_timeline(
    ising_circuit_fractional,
    idle_wires=False,
    target=backend_fractional.target,
    time_range=(0, 500),
    style=IQXSimple(),
)

<Image src="../docs/images/guides/fractional-gates/extracted-outputs/08dd1cdf-8b34-47c2-8324-f3538c9d1ab6-0.svg" alt="Output of the previous code cell" />

### Angle constraints

For the two-qubit $R_{ZZ}(\theta)$ gate, only angles between $0$ and $\pi/2$ can be executed on IBM Quantum hardware. If a circuit contains any $R_{ZZ}(\theta)$ gates with an angle outside of this range, then the standard transpilation pipeline will generally correct this with an appropriate circuit transformation (through the [`FoldRzzAngle`](../api/qiskit-ibm-runtime/transpiler-passes-fold-rzz-angle) pass).  However, for any $R_{ZZ}(\theta)$ gate containing one or more [`Parameter`](../api/qiskit/qiskit.circuit.Parameter)s, the transpiler will assume these parameters will be assigned angles within this range at runtime. The job will fail if any of the parameter values specified in the PUB submitted to Qiskit Runtime are outside of this range.

## Where to use fractional gates

Historically, the basis gates available on IBM Quantum QPUs have been **`CZ`**, **`X`**, **`RZ`**, **`SX`**, and **`ID`**, which can not efficiently represent circuits with single- and two-qubit rotations that are not multiples of $\pi / 2$. For example, an $R_X(\theta)$ gate, when transpiled, must decompose into a series of $RZ$ and $\sqrt{X}$ gates, which creates a circuit with two gates of finite duration instead of one.

Similarly, when two-qubit rotations such as an $R_{ZZ}(\theta)$ gate are transpiled, the decomposition requires two **`CZ`** gates and several single-qubit gates, which increases the circuit depth. These decompositions are shown in the following code.

In [5]:
qc = QuantumCircuit(1)
param = Parameter("θ")
qc.rx(param, 0)
qc.draw("mpl")

<Image src="../docs/images/guides/fractional-gates/extracted-outputs/11b003ee-aa8e-4794-a84a-b6870d15fa11-0.svg" alt="Output of the previous code cell" />

In [6]:
# Decomposition of an RX(θ) gate using the IBM Quantum QPU basis
service = QiskitRuntimeService()
backend = service.backend("ibm_torino")
optimization_level = 3
pm = generate_preset_pass_manager(optimization_level, backend=backend)
transpiled_circuit = pm.run(qc)
transpiled_circuit.draw("mpl", idle_wires=False)

<Image src="../docs/images/guides/fractional-gates/extracted-outputs/5da9bba4-9a3b-4569-9997-c5b9ccf87b6a-0.svg" alt="Output of the previous code cell" />

In [7]:
from qiskit import QuantumCircuit
from qiskit.circuit import Parameter
from qiskit.transpiler import generate_preset_pass_manager

from qiskit_ibm_runtime import QiskitRuntimeService

qc = QuantumCircuit(2)
param = Parameter("θ")
qc.rzz(param, 0, 1)
qc.draw("mpl")

<Image src="../docs/images/guides/fractional-gates/extracted-outputs/6b256201-0237-4f63-91ff-fde1bb884804-0.svg" alt="Output of the previous code cell" />

In [8]:
# Decomposition of an RZZ(θ) gate using the IBM Quantum QPU basis
service = QiskitRuntimeService()
backend = service.backend("ibm_torino")
optimization_level = 3
pm = generate_preset_pass_manager(optimization_level, backend=backend)
transpiled_circuit = pm.run(qc)
transpiled_circuit.draw("mpl", idle_wires=False)

<Image src="../docs/images/guides/fractional-gates/extracted-outputs/f07217b9-a6f0-4adf-b341-6da447535c33-0.svg" alt="Output of the previous code cell" />

For workflows that require many single-qubit $R_X(\theta)$ or two-qubit rotations (such as in a variational ansatz or when simulating the time evolution of quantum systems), this constraint causes the circuit depth to grow rapidly. However, fractional gates remove this requirement, because the single- and two-qubit rotations are executed directly, and create a more efficient (and thus error-suppressed) quantum circuit.

<span id="when-not-to-use"></span>
## When *not* to use fractional gates

It is important to note that fractional gates are an *experimental* feature and the behavior of the `use_fractional_gates` flag may change in the future. Look to the [release notes](/docs/api/qiskit-ibm-runtime/release-notes) for new versions of Qiskit Runtime for more information. See also the API reference documentation for [QiskitRuntimeService.backend](/docs/api/qiskit-ibm-runtime/qiskit-runtime-service#backend), which describes `use_fractional_gates`.

Additionally, the Qiskit transpiler has limited capability to use $R_{ZZ}(\theta)$ in its optimization passes. This requires you to take more care in crafting and optimizing circuits that contain these instructions.

Lastly, using fractional gates is not supported for:
- [Dynamic circuits](/docs/guides/classical-feedforward-and-control-flow)
- [Pauli twirling](/docs/guides/error-mitigation-and-suppression-techniques#pauli-twirling) - however, [measurement twirling with TREX](/docs/guides/error-mitigation-and-suppression-techniques#twirled-readout-error-extinction-trex) *is* supported.
- [Probabilistic error cancellation](/docs/guides/error-mitigation-and-suppression-techniques#probabilistic-error-cancellation-pec)
- [Zero-noise extrapolation (using probabilistic error amplification)](/docs/guides/error-mitigation-and-suppression-techniques#probabilistic-error-amplification-pea)

Read the guide on [primitive options](/docs/guides/runtime-options-overview) to learn more about customizing the error mitigation and suppression techniques for a given quantum workload.

## Next steps

<Admonition type="tip" title="Recommendations">
  -  To learn more about transpilation, see the [introduction to transpilation](/docs/guides/transpile) page.
  -  Read about [writing a custom transpiler pass](/docs/guides/custom-transpiler-pass).
  -  Understand how to [configure error mitigation](/docs/guides/configure-error-mitigation) for Qiskit Runtime.
</Admonition>